In [17]:
import pandas as pd

pest_name = '탄저병'

files = [
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_APPLE.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_BEAN.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_PEPPER.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_RICE.csv'
]

meta_cols = ['조사년도', '지역-시도', '지역-시군구', '지역-읍면동', '상세주소','좌표-경도', '좌표-위도', '조사회차', '조사일자(YYYYMMDD)']

all_data = []

for file in files:
    df = pd.read_csv(file)
    
    # "탄저병" 관련 컬럼만 추출
    pest_cols = [col for col in df.columns if pest_name in col]
    
    # 존재하는 메타 컬럼만 추려냄
    existing_meta = [col for col in meta_cols if col in df.columns]
    
    if pest_cols:
        selected = df[existing_meta + pest_cols].copy()

        # 지점 ID 부여 (위경도 기준)
        selected['지역-읍면동'] = selected['지역-읍면동'].fillna('미상')
        selected['지점ID'] = selected.groupby(
        ['지역-시도', '지역-시군구', '좌표-위도', '좌표-경도'],
        sort=False
        ).ngroup()


        all_data.append(selected)
        
result_df = pd.concat(all_data, ignore_index=True)
result_df = result_df.sort_values(by='지점ID', ascending=True)
cols = ['지점ID'] + [col for col in result_df.columns if col != '지점ID']
result_df = result_df[cols]
result_df.to_csv(f'{pest_name} sample6.csv', index=False)

In [22]:
import pandas as pd

pest_name = '탄저병'

files = [
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_APPLE.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_BEAN.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_PEPPER.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_RICE.csv'
]

meta_cols = ['조사년도', '지역-시도', '지역-시군구', '지역-읍면동', '상세주소', '좌표-경도', '좌표-위도', '조사회차', '조사일자(YYYYMMDD)']
all_data = []

for file in files:
    df = pd.read_csv(file)
    
    pest_cols = [col for col in df.columns if pest_name in col]
    existing_meta = [col for col in meta_cols if col in df.columns]
    
    if pest_cols:
        selected = df[existing_meta + pest_cols].copy()

        selected = selected.dropna(subset=['좌표-경도', '좌표-위도'])

        # 위경도를 float 그대로 소수점 10자리로 반올림해서 사용 (문자열 아님)
        selected['좌표-경도'] = selected['좌표-경도'].astype(float).round(10)
        selected['좌표-위도'] = selected['좌표-위도'].astype(float).round(10)

        selected['지점ID'] = selected.groupby(
            ['좌표-경도', '좌표-위도'], sort=False
        ).ngroup()


        all_data.append(selected)

result_df = pd.concat(all_data, ignore_index=True)
result_df = result_df.sort_values(by='지점ID', ascending=True)
cols = ['지점ID'] + [col for col in result_df.columns if col != '지점ID']
result_df = result_df[cols]
result_df.to_csv(f'{pest_name}_sample3.csv', index=False)


In [23]:
df_unique_coords = selected[['좌표-경도', '좌표-위도']].drop_duplicates()
print(len(df_unique_coords))
print(df_unique_coords)

2158
            좌표-경도      좌표-위도
0      129.231478  37.151701
6      129.227062  37.376098
12     129.103544  37.450083
18     129.102142  37.395499
24     128.896138  37.387742
...           ...        ...
23233  127.465426  36.857333
23239  127.505931  36.589967
23245  127.897179  36.950200
23510  126.292071  34.512724
23531  126.949771  35.023046

[2158 rows x 2 columns]


In [24]:
print(selected.dtypes[['좌표-경도', '좌표-위도']])


좌표-경도    float64
좌표-위도    float64
dtype: object


In [25]:
print(selected[['좌표-경도', '좌표-위도']].head(10).to_string(index=False))


     좌표-경도     좌표-위도
129.231478 37.151701
129.231478 37.151701
129.231478 37.151701
129.231478 37.151701
129.231478 37.151701
129.231478 37.151701
129.227062 37.376098
129.227062 37.376098
129.227062 37.376098
129.227062 37.376098


In [26]:
import pandas as pd

pest_name = '탄저병'

files = [
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_APPLE.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_BEAN.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_PEPPER.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_RICE.csv'
]

meta_cols = ['조사년도', '지역-시도', '지역-시군구', '지역-읍면동', '상세주소', '좌표-경도', '좌표-위도', '조사회차', '조사일자(YYYYMMDD)']
all_data = []

for file in files:
    df = pd.read_csv(file)
    
    pest_cols = [col for col in df.columns if pest_name in col]
    existing_meta = [col for col in meta_cols if col in df.columns]
    
    if pest_cols:
        selected = df[existing_meta + pest_cols].copy()
        selected = selected.dropna(subset=['좌표-경도', '좌표-위도'])

        # 좌표를 문자열로 변환 후 정확히 일치할 때만 동일 지점으로 처리
        selected['좌표-경도_str'] = selected['좌표-경도'].astype(str)
        selected['좌표-위도_str'] = selected['좌표-위도'].astype(str)

        selected['지점ID'] = selected.groupby(
            ['좌표-경도_str', '좌표-위도_str'], sort=False
        ).ngroup()

        all_data.append(selected)

result_df = pd.concat(all_data, ignore_index=True)
result_df = result_df.sort_values(by='지점ID', ascending=True)
cols = ['지점ID'] + [col for col in result_df.columns if col != '지점ID']
result_df = result_df[cols]
result_df.to_csv(f'{pest_name}_final_output.csv', index=False)


In [29]:
import pandas as pd

pest_name = '탄저병'

files = [
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_APPLE.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_BEAN.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_PEPPER.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_RICE.csv'
]

meta_cols = ['조사년도', '지역-시도', '지역-시군구', '지역-읍면동', '상세주소', '좌표-경도', '좌표-위도', '조사회차', '조사일자(YYYYMMDD)']
all_data = []

for file in files:
    df = pd.read_csv(file)
    
    pest_cols = [col for col in df.columns if pest_name in col]
    existing_meta = [col for col in meta_cols if col in df.columns]
    
    if pest_cols:
        selected = df[existing_meta + pest_cols].copy()
        selected = selected.dropna(subset=['좌표-경도', '좌표-위도'])
        all_data.append(selected)

# 모든 파일 병합 후 지점ID 부여
result_df = pd.concat(all_data, ignore_index=True)

# 소수점 10자리 반올림 → 문자열 변환
result_df['좌표-경도_str'] = result_df['좌표-경도'].astype(float).round(10).astype(str)
result_df['좌표-위도_str'] = result_df['좌표-위도'].astype(float).round(10).astype(str)

# 이제 전체에서 지점ID 부여
result_df['지점ID'] = result_df.groupby(
    ['좌표-경도_str', '좌표-위도_str'], sort=False
).ngroup()
result_df = result_df.drop(columns=['좌표-경도_str', '좌표-위도_str'])


# 정리
result_df = result_df.sort_values(by='지점ID', ascending=True)
cols = ['지점ID'] + [col for col in result_df.columns if col != '지점ID']
result_df = result_df[cols]
result_df.to_csv(f'{pest_name}_sample2.csv', index=False, encoding='utf-8-sig')


In [ ]:
import pandas as pd

pest_name = '탄저병'

files = [
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_APPLE.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_BEAN.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_PEPPER.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_RICE.csv'
]

meta_cols = ['조사년도', '지역-시도', '지역-시군구', '지역-읍면동',
             '상세주소', '좌표-경도', '좌표-위도', '조사회차', '조사일자(YYYYMMDD)']

all_data = []
for file in files:
    df = pd.read_csv(file)
    pest_cols = [col for col in df.columns if pest_name in col]
    existing_meta = [col for col in meta_cols if col in df.columns]

    if pest_cols:
        selected = df[existing_meta + pest_cols].copy()
        selected = selected.dropna(subset=['좌표-경도', '좌표-위도'])
        all_data.append(selected)

result_df = pd.concat(all_data, ignore_index=True)

law_path = "/Users/doyoung-gil/Desktop/연구실/법정동코드_전체자료.txt"
law_df = pd.read_csv(law_path, sep='\t', encoding='cp949', dtype={'법정동코드': str})

law_df = law_df[law_df['폐지여부'] == '존재'].copy()
law_df[['시도', '시군구', '읍면동']] = law_df['법정동명'].str.split(' ', n=2, expand=True)

result_df['지역-시군구'] = result_df['지역-시군구'].replace({'청원군': '청주시'})

merged = pd.merge(
    result_df,
    law_df[['법정동코드', '시도', '시군구', '읍면동']],
    left_on=['지역-시도', '지역-시군구', '지역-읍면동'],
    right_on=['시도', '시군구', '읍면동'],
    how='left'
)

missing = merged[merged['법정동코드'].isna()].copy()
law_df_sgg = law_df[law_df['읍면동'].isna()]  # 읍면동 없는 법정동만

backup = pd.merge(
    missing.drop(columns=['법정동코드', '시도', '시군구', '읍면동']),
    law_df_sgg[['법정동코드', '시도', '시군구']],
    left_on=['지역-시도', '지역-시군구'],
    right_on=['시도', '시군구'],
    how='left'
)

success = merged[merged['법정동코드'].notna()]
final_df = pd.concat([success, backup], ignore_index=True)

final_df['법정동코드'] = final_df['법정동코드'].astype(str).str.zfill(10)

final_df.drop(columns=['시도', '시군구', '읍면동'], errors='ignore', inplace=True)
final_df.rename(columns={'법정동코드': '지점ID'}, inplace=True)

final_df = final_df.sort_values(by=['지점ID', '조사년도'])

cols = ['지점ID'] + [col for col in final_df.columns if col != '지점ID']
final_df = final_df[cols]

final_df.to_csv(f"{pest_name}_sample3.csv", index=False, encoding='utf-8-sig')

In [ ]:
import pandas as pd
from pyproj import Transformer

# 1. 변환기 생성 (위경도 → EPSG:5181)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:5181", always_xy=True)

# 2. 격자 크기 설정 (단위: meter)
GRID_SIZE = 30  # 30m 격자

# 3. 예시 DataFrame (위경도 포함되어 있어야 함)
df = pd.read_csv("탄저병_sample3.csv")  # 기존 결과물 불러오기
df = df.dropna(subset=["좌표-경도", "좌표-위도"])

# 4. 위경도 → 5181 변환 및 격자 ID 생성
def generate_grid_id(lon, lat):
    x, y = transformer.transform(lon, lat)  # 위경도 → 평면좌표
    gx = int(x // GRID_SIZE)
    gy = int(y // GRID_SIZE)
    return f"{gx}_{gy}"

df["격자ID"] = df.apply(lambda row: generate_grid_id(row["좌표-경도"], row["좌표-위도"]), axis=1)

# 5. 지점ID를 격자ID로 교체
df = df.drop(columns=["지점ID"])
df = df.rename(columns={"격자ID": "지점ID"})
cols = ['지점ID'] + [col for col in df.columns if col != '지점ID']
df = df[cols]

df = df.sort_values(by='지점ID')

# 6. 저장
df.to_csv("탄저병_sample4.csv", index=False, encoding="utf-8-sig")


In [46]:
AWS_CSV = "/Users/doyoung-gil/Desktop/연구실/AIM_PEX_DT03_샘플데이터/KMA_AWS_DAILY.csv"
print(pd.read_csv(AWS_CSV, nrows=5).columns.tolist())

['stn_id', 'stn_nm', 'tm', 'lon', 'lat', 'ht', 'rn_day', 'tmax', 'tmin', 'wx_day']


In [48]:
check_CSV = "/Users/doyoung-gil/Desktop/연구실/탄저병_with_AWS_daily.csv"
print(pd.read_csv(check_CSV, nrows=5).columns.tolist())

['지점ID', 'date', 'nearest_stn_id', 'nearest_stn_name', 'dist_to_stn_m', '조사년도', '지역-시도', '지역-시군구', '지역-읍면동', '상세주소', '좌표-경도', '좌표-위도', '조사회차', '조사일자(YYYYMMDD)', '탄저병-병든과수/750과', '탄저병-병든과율(%)', '탄저병-병든꼬투리수(협/300협)', '탄저병-병든주율(%)', '탄저병-발병도', 'tmax', 'tmin', 'rain']


In [51]:
import pandas as pd
import numpy as np
from pyproj import Transformer
from scipy.spatial import cKDTree  # pip install scipy

DISEASE_CSV = "탄저병_sample4.csv"
AWS_CSV     = "/Users/doyoung-gil/Desktop/연구실/AIM_PEX_DT03_샘플데이터/KMA_AWS_DAILY.csv"

# 1) AWS 일자료 로드
usecols = ["stn_id","stn_nm","tm","lon","lat","ht","rn_day","tmax","tmin","wx_day"]
dtypes  = {
    "stn_id":"int32", "stn_nm":"string",
    "lon":"float32", "lat":"float32",
    "ht":"float32", "rn_day":"float32", "tmax":"float32","tmin":"float32",
    "wx_day":"float32" 
}
aws = pd.read_csv(AWS_CSV, usecols=usecols, dtype=dtypes, parse_dates=["tm"])
aws = aws.rename(columns={"tm":"date", "stn_nm":"stn_name"})

# 중복일 처리(안정성)
group_cols = ["stn_id","stn_name","date","lon","lat"]
agg = {"tmax":"mean","tmin":"mean","rn_day":"mean","ht":"mean","wx_day":"first"}
aws = aws.groupby(group_cols, as_index=False).agg(agg)

# 관측소 메타(좌표)
stn_meta = aws[["stn_id","stn_name","lon","lat"]].drop_duplicates("stn_id").reset_index(drop=True)

# 2) 샘플 로드
df = pd.read_csv(DISEASE_CSV).dropna(subset=["좌표-경도","좌표-위도"]).copy()
df["좌표-경도"] = df["좌표-경도"].astype(float)
df["좌표-위도"] = df["좌표-위도"].astype(float)
df["date"] = pd.to_datetime(df["조사일자(YYYYMMDD)"].astype(str), format="%Y%m%d", errors="coerce")

# 3) 좌표 변환(5181) & 최근접 관측소 매칭(거리 컬럼은 저장 안 함)
to_5181 = Transformer.from_crs("EPSG:4326","EPSG:5181", always_xy=True)
stn_xy = np.column_stack(to_5181.transform(stn_meta["lon"].values, stn_meta["lat"].values))
site_xy = np.column_stack(to_5181.transform(df["좌표-경도"].values, df["좌표-위도"].values))

idx = cKDTree(stn_xy).query(site_xy, k=1)[1]  # 인덱스만 사용
df["nearest_stn_id"]   = stn_meta.loc[idx, "stn_id"].values
df["nearest_stn_name"] = stn_meta.loc[idx, "stn_name"].values

# 4) 날짜로 기상 붙이기
weather_cols = ["stn_id","date","ht","rn_day","tmax","tmin","wx_day"]
df_merged = df.merge(
    aws[weather_cols],
    left_on=["nearest_stn_id","date"],
    right_on=["stn_id","date"],
    how="left"
).drop(columns=["stn_id", "date"])  # date도 제거

# 5) 조사일자 바로 뒤에 기상 컬럼 배치
wx_bundle = ["nearest_stn_id","nearest_stn_name","ht","rn_day","tmax","tmin","wx_day"]
cols = df_merged.columns.tolist()
pos = cols.index("조사일자(YYYYMMDD)")
head = cols[:pos+1]  # '... , 조사일자(YYYYMMDD)'
used = set(head) | set(wx_bundle) | {"date"}  # 이미 쓴 것 + 지울 것
tail = [c for c in cols if c not in used]

df_merged = df_merged[head + wx_bundle + tail]

# 보기 좋게 정렬
if "지점ID" in df_merged.columns:
    df_merged = df_merged.sort_values(["지점ID","조사일자(YYYYMMDD)"]).reset_index(drop=True)

# 6) 저장
df_merged.to_csv("탄저병_with_AWS_daily.csv", index=False, encoding="utf-8-sig")
print("완료 → 탄저병_with_AWS_daily.csv")
print(df_merged.head(3))


완료 → 탄저병_with_AWS_daily.csv
         지점ID  조사년도 지역-시도 지역-시군구 지역-읍면동     상세주소       좌표-경도      좌표-위도  조사회차  \
0  10000_6494  2016  경상남도    진주시    집현면  냉정리 377  128.098696  35.245024     1   
1  10000_6494  2016  경상남도    진주시    집현면  냉정리 377  128.098696  35.245024     1   
2  10000_6494  2016  경상남도    진주시    집현면  냉정리 377  128.098696  35.245024     2   

   조사일자(YYYYMMDD)  ...  ht rn_day  tmax  tmin  wx_day  탄저병-병든과수/750과  \
0        20160701  ... NaN    NaN   NaN   NaN     NaN            NaN   
1        20160701  ... NaN    NaN   NaN   NaN     NaN            NaN   
2        20160718  ... NaN    NaN   NaN   NaN     NaN            NaN   

   탄저병-병든과율(%)  탄저병-병든꼬투리수(협/300협)  탄저병-병든주율(%)  탄저병-발병도  
0          NaN                 NaN          NaN      NaN  
1          NaN                 0.0          0.0      NaN  
2          NaN                 0.0          0.0      NaN  

[3 rows x 22 columns]
